<a href="https://colab.research.google.com/github/1FATIMAH1/IT362/blob/main/Phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1: Data Collection

## Real Estate Analysis in Saudi Arabia

**Research question:** In Riyadh, how do neighborhood and property type relate to average annual residential rental prices?

The data was collected directly from the Saudi real-estate website SATEL using its API. The data was not downloaded or uploaded as a ready-made dataset.

**Website:** [SATEL Property Listings](https://listings.satel.sa/)

**API used for data collection:** https://apiv2.satel.sa/property/filter?limit=500

In [1]:
# Import the libraries
import requests
import json
import pandas as pd

In [2]:
# The API used by the Saudi SATEL real-estate website
url = 'https://apiv2.satel.sa/property/filter?limit=500'
collection_date = '2026-09-07'

print ('Source:', url)
print ('Collection date:', collection_date)

Source: https://apiv2.satel.sa/property/filter?limit=500
Collection date: 2026-09-07


In [3]:
# Send a request to the API
response = requests.get(url, timeout=30)

print ('Status code:', response.status_code)

Status code: 200


In [4]:
# Convert the API response from JSON to a Python list
if response.status_code == 200:
    raw_data = response.json()
    print ('The data was collected successfully!')
    print ('Number of raw records:', len(raw_data))
else:
    print ('The data was not collected.')

The data was collected successfully!
Number of raw records: 226


In [5]:
# Save the original raw data without any modification
with open('raw_satel_riyadh_rentals.json', 'w', encoding='utf-8') as f:
    json.dump(raw_data, f, ensure_ascii=False, indent=4)

print ('The raw JSON file was saved.')

The raw JSON file was saved.


### The unstructured part of the raw data

In [6]:
property_id = []
neighborhood = []
neighborhood_ar = []
property_type = []
annual_rent_sar = []
area_sqm = []
bedrooms = []
bathrooms = []
furnishing = []
status = []
title_text = []
created_at = []

In [7]:
# Select residential annual rental records in Riyadh
for item in raw_data:
    if item.get('type') == 'Rent' and item.get('catName') == 'Residential' and item.get('priceGroup') == 'annual':
        property_id.append(item.get('propertyNumber'))
        neighborhood.append(item.get('subCityEn'))
        neighborhood_ar.append(item.get('subCityAr'))
        property_type.append(item.get('subCatName'))
        annual_rent_sar.append(item.get('price'))
        area_sqm.append(item.get('floorArea'))
        bedrooms.append(item.get('beds'))
        bathrooms.append(item.get('baths'))
        furnishing.append(item.get('furnishing'))
        status.append(item.get('status'))
        title_text.append(item.get('titleEn'))
        created_at.append(item.get('createdAt'))

print ('Number of selected annual residential rentals:', len(property_id))

Number of selected annual residential rentals: 217


In [8]:
dictionary = {
    'property_id': property_id,
    'neighborhood': neighborhood,
    'neighborhood_ar': neighborhood_ar,
    'property_type': property_type,
    'annual_rent_sar': annual_rent_sar,
    'area_sqm': area_sqm,
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'furnishing': furnishing,
    'status': status,
    'title_text': title_text,
    'created_at': created_at,
    'source_url': [url] * len(property_id),
    'collection_date': [collection_date] * len(property_id)
}

df = pd.DataFrame(dictionary)
print (df.head())

  property_id neighborhood neighborhood_ar property_type  annual_rent_sar  \
0       C0055    An Narjis          النرجس     Compounds           110000   
1       C0087     Al Olaya          العليا    Apartments           160000   
2       A0177    An Narjis          النرجس     Compounds           140000   
3       C0052     Al Malaz        حي الملز     Compounds            73000   
4       C0084   An Nakheel       حي النخيل        Villas           370000   

   area_sqm  bedrooms  bathrooms           furnishing      status  \
0     139.0         3          4          Unfurnished   Available   
1     112.0         2          3      Fully furnished   Available   
2     132.0         3          3      Fully furnished   Available   
3      95.0         2          2      Fully furnished  Rented out   
4     460.0         6          6  Partially furnished  Rented out   

                                          title_text  \
0  Satel at Opal on Thoumamah Road - 3 Bedrooms A...   
1         

In [9]:
print ('Number of observations:', len(df))
print ('Number of features:', len(df.columns))
print (df.dtypes)

Number of observations: 217
Number of features: 14
property_id         object
neighborhood        object
neighborhood_ar     object
property_type       object
annual_rent_sar      int64
area_sqm           float64
bedrooms             int64
bathrooms            int64
furnishing          object
status              object
title_text          object
created_at          object
source_url          object
collection_date     object
dtype: object


In [10]:
# Check missing values
print (df.isnull().sum())

property_id        0
neighborhood       0
neighborhood_ar    0
property_type      0
annual_rent_sar    0
area_sqm           0
bedrooms           0
bathrooms          0
furnishing         0
status             0
title_text         0
created_at         0
source_url         0
collection_date    0
dtype: int64


In [11]:
# Show how many records were collected from each neighborhood
print (df['neighborhood'].value_counts())

# Show how many records were collected for each property type
print (df['property_type'].value_counts())

neighborhood
Al Olaya                    40
An Narjis                   19
Al Malqa                    15
Al Wurud                    14
An Nakheel                  12
Al Aqiq                     12
Al Mathar Ash Shamali       12
Al Malaz                    10
Ar Rahmaniyyah               8
As Sulimaniyah               8
Al Mohammadiyyah             8
Alyasmin                     7
As Sahafah                   6
Hittin                       5
At Taawun                    5
Ar Rafiah                    4
Qurtubah                     4
King Salman Neighborhood     2
Al Ghadir                    2
Al Murabba                   2
Al Muruj                     2
Sedra                        2
Al Aarid                     2
Al Qirawan                   2
Irqah                        1
Almasiaf                     1
King Fahd                    1
Ar Rawdah                    1
Jarir                        1
An Nada                      1
An Nuzhah                    1
Al Wadi                   

In [12]:
# Calculate the average annual rent for each neighborhood and property type
average_rent = df[['neighborhood', 'property_type', 'annual_rent_sar']].groupby(['neighborhood', 'property_type']).mean()
print (average_rent)

                                        annual_rent_sar
neighborhood             property_type                 
Aarid                    Apartments        90000.000000
Al Aarid                 Apartments        95000.000000
                         Villas           145000.000000
Al Aqiq                  Apartments       107300.000000
                         Duplex           177500.000000
...                                                 ...
King Salman Neighborhood Apartments        80000.000000
Qurtubah                 Apartments        93333.333333
                         Villas           280000.000000
Sedra                    Villas           235000.000000
حي الملقا                Apartments       125000.000000

[65 rows x 1 columns]


In [13]:
# Save the collected table
df.to_csv('riyadh_annual_rentals_collected.csv', index=False, encoding='utf-8-sig')

print ('The collected CSV file was saved.')

The collected CSV file was saved.
